In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

import duckdb
import pandas as pd
import pyarrow.parquet as pq

# Remount Google Drive cleanly.
try:
    drive.flush_and_unmount()
except Exception:
    pass

if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

drive.mount(
    "/content/drive",
    force_remount=True,
    timeout_ms=300000,
)

MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.exists():
    raise RuntimeError("MyDrive is not available after mounting.")

DATA_DIR = MY_DRIVE / "Language Detection"
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {DATA_DIR}")

PARQUET_PATH = (
    DATA_DIR
    / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
)

if not PARQUET_PATH.is_file():
    available_files = [
        path.name
        for path in DATA_DIR.iterdir()
        if path.is_file()
    ]
    raise FileNotFoundError(
        f"Parquet file not found: {PARQUET_PATH}\n"
        f"Available files: {available_files}"
    )

parquet_file = pq.ParquetFile(PARQUET_PATH)
metadata = parquet_file.metadata

print("File path  :", PARQUET_PATH)
print("File size  :", f"{PARQUET_PATH.stat().st_size / (1024**2):.2f} MB")
print("Rows       :", f"{metadata.num_rows:,}")
print("Row groups :", metadata.num_row_groups)
print("Columns    :", metadata.num_columns)


Mounted at /content/drive
File path  : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
File size  : 469.49 MB
Rows       : 3,469
Row groups : 1
Columns    : 12


In [ ]:
SEGMENT_LIMIT = 25
con = duckdb.connect()


def get_language_review(language_code, segment_limit=SEGMENT_LIMIT):
    """Return the first transcript segments from the latest session for a language."""
    review_df = con.execute(
        f"""
        WITH latest_session AS (
            SELECT
                gamesession_id,
                url,
                TRY_CAST(created_at AS TIMESTAMP) AS created_at,
                lang_detected,
                transcript_segments
            FROM read_parquet('{PARQUET_PATH.as_posix()}')
            WHERE
                lang_detected = ?
                AND transcript_segments IS NOT NULL
            ORDER BY
                TRY_CAST(created_at AS TIMESTAMP) DESC,
                gamesession_id DESC
            LIMIT 1
        ),
        exploded AS (
            SELECT
                gamesession_id,
                url,
                lang_detected,
                generate_subscripts(transcript_segments, 1) AS segment_index,
                UNNEST(transcript_segments) AS segment
            FROM latest_session
        ),
        segments AS (
            SELECT
                gamesession_id,
                url,
                segment_index,
                TRY_CAST(segment.timestamp[1] AS DOUBLE) AS start_seconds,
                TRY_CAST(segment.timestamp[2] AS DOUBLE) AS end_seconds,
                TRIM(segment.text) AS segment_text,
                lang_detected AS language_detected
            FROM exploded
            WHERE
                segment.text IS NOT NULL
                AND TRIM(segment.text) <> ''
        )
        SELECT
            gamesession_id,
            url,
            segment_index,
            printf(
                '%02d:%02d:%02d - %02d:%02d:%02d',
                CAST(FLOOR(start_seconds / 3600) AS INTEGER),
                CAST(FLOOR(start_seconds / 60) % 60 AS INTEGER),
                CAST(FLOOR(start_seconds) % 60 AS INTEGER),
                CAST(FLOOR(end_seconds / 3600) AS INTEGER),
                CAST(FLOOR(end_seconds / 60) % 60 AS INTEGER),
                CAST(FLOOR(end_seconds) % 60 AS INTEGER)
            ) AS segment_timestamp,
            segment_text,
            language_detected
        FROM segments
        WHERE
            start_seconds IS NOT NULL
            AND end_seconds IS NOT NULL
        ORDER BY segment_index ASC
        LIMIT ?
        """,
        [language_code, segment_limit],
    ).df()

    # String dtype is intentional: verdicts can contain notes such as
    # "Correct", "No speech", "Missing word: ...", etc.
    review_df["VERDICT"] = pd.Series(
        pd.NA,
        index=review_df.index,
        dtype="string",
    )
    return review_df


def show_review(review_df):
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
    ):
        display(review_df)


In [ ]:
english_review = get_language_review("en")

english_manual_review = {
    "00:00:02 - 00:00:02": "No speech",
    "00:00:14 - 00:00:15": "Wrong words: Don't be turning around, shut my door.",
    "00:01:07 - 00:01:08": "Correct",
    "00:02:23 - 00:02:25": "Correct",
    "00:02:46 - 00:02:49": "Correct",
    "00:02:50 - 00:02:54": "Missing words: nothing personal.",
    "00:02:55 - 00:02:59": "Missing word: just.",
    "00:02:59 - 00:03:06": "Wrong words: Okay so, just shout.",
    "00:03:20 - 00:03:26": "Correct",
    "00:03:27 - 00:03:28": "Correct",
    "00:03:44 - 00:03:47": "Correct",
    "00:03:48 - 00:03:55": "Wrong words: the devil's; symptom logged.",
    "00:03:55 - 00:03:58": "Correct",
    "00:04:05 - 00:04:08": "Correct",
    "00:04:08 - 00:04:12": "Correct",
    "00:04:12 - 00:04:16": "Correct",
    "00:04:17 - 00:04:20": "Correct",
    "00:04:21 - 00:04:24": "Missing word: and.",
    "00:04:25 - 00:04:29": "Correct",
    "00:04:29 - 00:04:34": "Extra words: I'm; missing word: but.",
    "00:04:35 - 00:04:38": "Extra word: What's.",
    "00:05:31 - 00:05:34": "Missing words: What's the point.",
    "00:05:43 - 00:05:46": "Correct",
    "00:05:47 - 00:05:52": "Correct",
    "00:05:54 - 00:05:59": "Correct",
}

english_review["VERDICT"] = (
    english_review["segment_timestamp"]
    .map(english_manual_review)
    .astype("string")
)

if english_review["VERDICT"].isna().any():
    missing = english_review.loc[
        english_review["VERDICT"].isna(),
        "segment_timestamp",
    ].tolist()
    raise ValueError(f"Missing English manual review: {missing}")

show_review(english_review)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268501,https://www.twitch.tv/videos/2854311727,1,00:00:02 - 00:00:02,We'll,en,No speech
1,141268501,https://www.twitch.tv/videos/2854311727,2,00:00:14 - 00:00:15,"Don't be talking, shut my door.",en,"Wrong words: Don't be turning around, shut my door."
2,141268501,https://www.twitch.tv/videos/2854311727,3,00:01:07 - 00:01:08,We'll start it.,en,Correct
3,141268501,https://www.twitch.tv/videos/2854311727,4,00:02:23 - 00:02:25,We'll start with the basics.,en,Correct
4,141268501,https://www.twitch.tv/videos/2854311727,5,00:02:46 - 00:02:49,actually showed up. You guys owe me ten bucks.,en,Correct
5,141268501,https://www.twitch.tv/videos/2854311727,6,00:02:50 - 00:02:54,"personal. It's just, uh, we're stuck in Groundhog Day out here.",en,Missing words: nothing personal.
6,141268501,https://www.twitch.tv/videos/2854311727,7,00:02:55 - 00:02:59,"trying to, you know, not go crazy..",en,Missing word: just.
7,141268501,https://www.twitch.tv/videos/2854311727,8,00:02:59 - 00:03:06,"Okay, it's simple really, shout next and the guys will let the survivor in.",en,"Wrong words: Okay so, just shout."
8,141268501,https://www.twitch.tv/videos/2854311727,9,00:03:20 - 00:03:26,"Next survivor. First, do a simple inspection.",en,Correct
9,141268501,https://www.twitch.tv/videos/2854311727,10,00:03:27 - 00:03:28,Take the flashlight from the table.,en,Correct


In [11]:
import duckdb
import pandas as pd

LANGUAGE = "en"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

english_segments = con.execute(
    f"""
    WITH latest_session AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected = '{LANGUAGE}'
            AND transcript_segments IS NOT NULL
        ORDER BY
            TRY_CAST(created_at AS TIMESTAMP) DESC,
            gamesession_id DESC
        LIMIT 1
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM latest_session
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            lang_detected AS language_detected
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
            AND len(segment.words) >= {MIN_WORDS}
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        language_detected
    FROM eligible
    WHERE
        segment_start IS NOT NULL
        AND segment_end IS NOT NULL
    ORDER BY segment_index ASC
    LIMIT {SEGMENTS_NEEDED}
    """
).df()


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


english_segments["segment_timestamp"] = (
    english_segments["segment_start"].map(format_timestamp)
    + " - "
    + english_segments["segment_end"].map(format_timestamp)
)

english_segments["VERDICT"] = ""

english_segments = english_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(english_segments)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268501,https://www.twitch.tv/videos/2854311727,2,00:00:14 - 00:00:15,"Don't be talking, shut my door.",en,
1,141268501,https://www.twitch.tv/videos/2854311727,4,00:02:23 - 00:02:25,We'll start with the basics.,en,
2,141268501,https://www.twitch.tv/videos/2854311727,5,00:02:46 - 00:02:49,actually showed up. You guys owe me ten bucks.,en,
3,141268501,https://www.twitch.tv/videos/2854311727,6,00:02:50 - 00:02:54,"personal. It's just, uh, we're stuck in Groundhog Day out here.",en,
4,141268501,https://www.twitch.tv/videos/2854311727,7,00:02:55 - 00:02:59,"trying to, you know, not go crazy..",en,
5,141268501,https://www.twitch.tv/videos/2854311727,8,00:02:59 - 00:03:06,"Okay, it's simple really, shout next and the guys will let the survivor in.",en,
6,141268501,https://www.twitch.tv/videos/2854311727,9,00:03:20 - 00:03:26,"Next survivor. First, do a simple inspection.",en,
7,141268501,https://www.twitch.tv/videos/2854311727,10,00:03:27 - 00:03:28,Take the flashlight from the table.,en,
8,141268501,https://www.twitch.tv/videos/2854311727,11,00:03:44 - 00:03:47,Take your time. Match what you see against the symptom chart.,en,
9,141268501,https://www.twitch.tv/videos/2854311727,12,00:03:48 - 00:03:55,devil's in the details. Identify the particular ailment and click symptom locked.,en,


In [ ]:
german_review = get_language_review("de")

# Fill verdicts after manual listening. Examples:
# german_review.loc[0, "VERDICT"] = "Correct"
# german_review.loc[1, "VERDICT"] = "Missing word: ..."
# german_review.loc[2, "VERDICT"] = "Wrong words: ..."
# german_review.loc[3, "VERDICT"] = "Extra word: ..."
# german_review.loc[4, "VERDICT"] = "No speech"

show_review(german_review)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141192575,https://www.twitch.tv/videos/2852368077,1,00:00:07 - 00:00:10,"Der kann mich gar nicht erwischt, nie im Leben.",de,<NA>
1,141192575,https://www.twitch.tv/videos/2852368077,2,00:00:14 - 00:00:15,ich hab's gerade gesehen.,de,<NA>
2,141192575,https://www.twitch.tv/videos/2852368077,3,00:00:25 - 00:00:27,"Eigentlich hat das Gebäude, als wir offiziell in der Zwiebel waren.",de,<NA>
3,141192575,https://www.twitch.tv/videos/2852368077,4,00:00:46 - 00:00:53,"Ich seh' halt nicht, keine Ahnung, wo der ist.",de,<NA>
4,141192575,https://www.twitch.tv/videos/2852368077,5,00:00:59 - 00:01:04,"Irgendwo Richtung Turm drin, irgendwo da hinten, so ein bisschen weiter links.",de,<NA>
5,141192575,https://www.twitch.tv/videos/2852368077,6,00:01:08 - 00:01:16,"einen haben wir am Tachel, einer ist noch unten drin ja ja da snipet er die Lachen",de,<NA>
6,141192575,https://www.twitch.tv/videos/2852368077,7,00:01:19 - 00:01:28,"traurige ja gib mir ein Bredi ja ein richtiger Lack jetzt habe ich gepisst das, ich hätte dich geholt",de,<NA>
7,141192575,https://www.twitch.tv/videos/2852368077,8,00:01:30 - 00:01:41,"ich weiß nicht, hier haben die die selber gibt's das das war das, was du deinen eigenen Strömen liebst warte mal ich schau mal Das ist jetzt Beste.",de,<NA>
8,141192575,https://www.twitch.tv/videos/2852368077,9,00:01:42 - 00:01:43,Hinter uns.,de,<NA>
9,141192575,https://www.twitch.tv/videos/2852368077,10,00:02:06 - 00:02:13,"Die hat ja noch einer, jetzt können sie Tank und Psych nochmal.",de,<NA>


In [ ]:
russian_review = get_language_review("ru")

# Fill verdicts after manual listening. Examples:
# russian_review.loc[0, "VERDICT"] = "Correct"
# russian_review.loc[1, "VERDICT"] = "Missing word: ..."
# russian_review.loc[2, "VERDICT"] = "Wrong words: ..."
# russian_review.loc[3, "VERDICT"] = "Extra word: ..."
# russian_review.loc[4, "VERDICT"] = "No speech"

show_review(russian_review)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,1,00:00:09 - 00:00:39,ДИНАМИЧНАЯ,ru,<NA>
1,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,2,00:01:00 - 00:01:30,"ДИНАМИЧНАЯ Ooh-ooh, kiss my shit, kiss my",ru,<NA>
2,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,3,00:01:30 - 00:01:44,"shit I heard you were talking shit And you didn't think that I would hear it People hear you talking, like",ru,<NA>
3,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,4,00:03:18 - 00:03:20,в,ru,<NA>
4,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,5,00:03:20 - 00:03:50,ДИНАМИЧНАЯ,ru,<NA>
5,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,6,00:03:50 - 00:03:54,"Так я не понимаю, нахуй он сначала девочек за бодашкой врубил?",ru,<NA>
6,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,7,00:03:57 - 00:04:04,"Чад, всем дарова, всем... Ты что, реально врубил, придурок? Блять, ты ебла нахуй, нахуй ты меня трахаешь, ебанашка Ростян, он реально врубил, Ростян?",ru,<NA>
7,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,8,00:04:04 - 00:04:26,"Да, я врубил Да Короче, дарова, Чад, привет, всем Ну вот, так вам скажу, нас сегодня распаковка анимешных фигурок, анимешных фигурок из магазина popo-man или popo-ma popo-me-me-me вот",ru,<NA>
8,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,9,00:04:28 - 00:04:56,"а, оффот? пишут, пишут, оффот, не надо стримить всё, тогда оффо, нахуй, чад, спасибо, всем пока короче, взяли конструктивного компотика, ну, такого рационального правильного, вкусного собрались в небольшой компании рациональных людей это окей Юра окей вот Юра вот Леша вот",ru,<NA>
9,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,10,00:04:57 - 00:05:26,Рациональный Леша нас сегодня Леша нас сегодня диджей Леша диджей и Растян с танком Растян с танком я тебя могу вот что будет так что то попьем пивка распакуем игрушки и ляжем спать вот все не знаю кто спать сегодня денечек такой не сбылся,ru,<NA>


In [ ]:
french_review = get_language_review("fr")

# Fill verdicts after manual listening. Examples:
# french_review.loc[0, "VERDICT"] = "Correct"
# french_review.loc[1, "VERDICT"] = "Missing word: ..."
# french_review.loc[2, "VERDICT"] = "Wrong words: ..."
# french_review.loc[3, "VERDICT"] = "Extra word: ..."
# french_review.loc[4, "VERDICT"] = "No speech"

show_review(french_review)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141267561,https://www.twitch.tv/videos/2854116559,1,00:00:29 - 00:00:29,serai là-bas,fr,<NA>
1,141267561,https://www.twitch.tv/videos/2854116559,2,00:00:52 - 00:00:55,What you guys doing ?,fr,<NA>
2,141267561,https://www.twitch.tv/videos/2854116559,3,00:01:24 - 00:01:25,Mais quoi,fr,<NA>
3,141267561,https://www.twitch.tv/videos/2854116559,4,00:01:42 - 00:01:43,? Je serai,fr,<NA>
4,141267561,https://www.twitch.tv/videos/2854116559,5,00:02:15 - 00:02:16,là-bas,fr,<NA>
5,141267561,https://www.twitch.tv/videos/2854116559,6,00:02:38 - 00:02:41,What you guys doing ?,fr,<NA>
6,141267561,https://www.twitch.tv/videos/2854116559,7,00:03:06 - 00:03:14,"Allez mec Attendez Yo Damien, t le premier ?",fr,<NA>
7,141267561,https://www.twitch.tv/videos/2854116559,8,00:03:14 - 00:03:24,"Ouais sur tiktak t le premier, même sur Twitch je pense En tout cas le premier message Euh, y'en perd pas de temps Aujourd'hui on essaie d'avancer putain de capa de merde Comme ça c'est fait, il nous manquera quoi ?",fr,<NA>
8,141267561,https://www.twitch.tv/videos/2854116559,9,00:03:24 - 00:03:34,"Là, il'on y s'extraire avec l'autre un, ça nous donnera la peluche, et après il manquera juste l'achantique, et problème de l'achantique, c que je sais pas comment je vais le pour la trouver, mais on va tenter, on va tenter.",fr,<NA>
9,141267561,https://www.twitch.tv/videos/2854116559,10,00:03:34 - 00:03:38,"Yo Pilouf,'as va ou quoi ? Comment va le Pilouf ?",fr,<NA>


In [ ]:
spanish_review = get_language_review("es")

# Fill verdicts after manual listening. Examples:
# spanish_review.loc[0, "VERDICT"] = "Correct"
# spanish_review.loc[1, "VERDICT"] = "Missing word: ..."
# spanish_review.loc[2, "VERDICT"] = "Wrong words: ..."
# spanish_review.loc[3, "VERDICT"] = "Extra word: ..."
# spanish_review.loc[4, "VERDICT"] = "No speech"

show_review(spanish_review)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268448,https://www.twitch.tv/videos/2853628924,1,00:03:03 - 00:03:14,hola hola que tal gente silla estamos aquí otra vez un día más vamos al charlando,es,<NA>
1,141268448,https://www.twitch.tv/videos/2853628924,2,00:03:17 - 00:03:47,hello como estáis como estáis ahora estoy bastante cansada pero tengo una cosa aquí estamos aquí cumplimos vale pues hoy vamos a jugar a los dinosaurios que me apetece un ratito aunque sea y ya luego vamos al balo en un ratito hoy no me quedaré mucho porque me apetece estar más en la gama todo esto me hecho una siesta cuando he llegado,es,<NA>
2,141268448,https://www.twitch.tv/videos/2853628924,3,00:03:47 - 00:03:59,de tres horas y claro una mañana que hay que trabajar entonces prefiero irme tempranito que sabemos como acaba de irse tempranito pero bueno lo cuento igual.,es,<NA>
3,141268448,https://www.twitch.tv/videos/2853628924,4,00:04:02 - 00:04:05,eh... Vale. ¿Dónde está? ¿Dónde está?,es,<NA>
4,141268448,https://www.twitch.tv/videos/2853628924,5,00:04:05 - 00:04:06,"Aquí,",es,<NA>
5,141268448,https://www.twitch.tv/videos/2853628924,6,00:04:11 - 00:04:13,"Let's go. Vamos allá, continúen.",es,<NA>
6,141268448,https://www.twitch.tv/videos/2853628924,7,00:04:18 - 00:04:21,"Vale, voy limpiar un momentito rápido el teclado.",es,<NA>
7,141268448,https://www.twitch.tv/videos/2853628924,8,00:04:30 - 00:04:33,Vale. Vale.,es,<NA>
8,141268448,https://www.twitch.tv/videos/2853628924,9,00:04:38 - 00:04:49,"Pero no lo que tenía que hacer, tipo, las dos canastas y los ladrillos de piedra vale,",es,<NA>
9,141268448,https://www.twitch.tv/videos/2853628924,10,00:04:52 - 00:05:03,"ehm, los de piedra y las dos canastas, tengo que hacer nooo, una planta por la puta,",es,<NA>


In [ ]:
portuguese_review = get_language_review("pt")

# Fill verdicts after manual listening. Examples:
# portuguese_review.loc[0, "VERDICT"] = "Correct"
# portuguese_review.loc[1, "VERDICT"] = "Missing word: ..."
# portuguese_review.loc[2, "VERDICT"] = "Wrong words: ..."
# portuguese_review.loc[3, "VERDICT"] = "Extra word: ..."
# portuguese_review.loc[4, "VERDICT"] = "No speech"

show_review(portuguese_review)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268410,https://www.twitch.tv/videos/2854267348,1,00:09:52 - 00:09:59,FUERTA DE RAHIM Ta,pt,<NA>
1,141268410,https://www.twitch.tv/videos/2854267348,2,00:10:54 - 00:11:10,de pé E Fui FUERTA RAHIM E ASSISTA FUERTA FAZ FUERTA RAHIM FUERTA RAHIM,pt,<NA>
2,141268410,https://www.twitch.tv/videos/2854267348,3,00:11:13 - 00:11:19,FUERTA RAHIM porta aberta e isso,pt,<NA>
3,141268410,https://www.twitch.tv/videos/2854267348,4,00:11:23 - 00:11:47,ok então estou bebendo um café hoje seria o dia do chill stream mas essa semana eu não joguei a LoL eu queria jogar a LoL eu queria jogar a LoL o chill stream significa que eu decido então se eu quero jogar LoL eu a LoL e vocês pegam no cu aquele eu mais alto talvez,pt,<NA>
4,141268410,https://www.twitch.tv/videos/2854267348,5,00:11:54 - 00:11:57,Como você Como você Galaxi?,pt,<NA>
5,141268410,https://www.twitch.tv/videos/2854267348,6,00:11:58 - 00:12:02,"Você fazendo tudo, você está certo. Como você Pasta?",pt,<NA>
6,141268410,https://www.twitch.tv/videos/2854267348,7,00:12:02 - 00:12:09,"Como você está? Shadow? Tudo bem, garotos? Razy, como você Tchau garotos, tchau a você bem?",pt,<NA>
7,141268410,https://www.twitch.tv/videos/2854267348,8,00:12:14 - 00:12:18,"O Urco desde esta manhã, oh Deus, desta manhã...",pt,<NA>
8,141268410,https://www.twitch.tv/videos/2854267348,9,00:12:18 - 00:12:21,"Eu não estava. E saiu também no meu schedule, não sei se você viu.",pt,<NA>
9,141268410,https://www.twitch.tv/videos/2854267348,10,00:12:22 - 00:12:28,"Que é feito, não mais a Dorari. Mas é a...",pt,<NA>


In [ ]:
all_reviews = pd.concat(
    [
        english_review,
        german_review,
        russian_review,
        french_review,
        spanish_review,
        portuguese_review,
    ],
    ignore_index=True,
)

show_review(all_reviews)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268501,https://www.twitch.tv/videos/2854311727,1,00:00:02 - 00:00:02,We'll,en,No speech
1,141268501,https://www.twitch.tv/videos/2854311727,2,00:00:14 - 00:00:15,"Don't be talking, shut my door.",en,"Wrong words: Don't be turning around, shut my door."
2,141268501,https://www.twitch.tv/videos/2854311727,3,00:01:07 - 00:01:08,We'll start it.,en,Correct
3,141268501,https://www.twitch.tv/videos/2854311727,4,00:02:23 - 00:02:25,We'll start with the basics.,en,Correct
4,141268501,https://www.twitch.tv/videos/2854311727,5,00:02:46 - 00:02:49,actually showed up. You guys owe me ten bucks.,en,Correct
5,141268501,https://www.twitch.tv/videos/2854311727,6,00:02:50 - 00:02:54,"personal. It's just, uh, we're stuck in Groundhog Day out here.",en,Missing words: nothing personal.
6,141268501,https://www.twitch.tv/videos/2854311727,7,00:02:55 - 00:02:59,"trying to, you know, not go crazy..",en,Missing word: just.
7,141268501,https://www.twitch.tv/videos/2854311727,8,00:02:59 - 00:03:06,"Okay, it's simple really, shout next and the guys will let the survivor in.",en,"Wrong words: Okay so, just shout."
8,141268501,https://www.twitch.tv/videos/2854311727,9,00:03:20 - 00:03:26,"Next survivor. First, do a simple inspection.",en,Correct
9,141268501,https://www.twitch.tv/videos/2854311727,10,00:03:27 - 00:03:28,Take the flashlight from the table.,en,Correct


In [10]:
import duckdb
import pandas as pd

LANGUAGE = "en"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

english_segments = con.execute(
    f"""
    WITH source AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE lang_detected = '{LANGUAGE}'
          AND transcript_segments IS NOT NULL
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            UNNEST(transcript_segments) AS segment
        FROM source
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            segment.text AS segment_text,
            segment.timestamp[1] AS segment_start,
            segment.timestamp[2] AS segment_end
        FROM exploded
        WHERE segment.text IS NOT NULL
          AND len(segment.words) >= {MIN_WORDS}
    ),
    latest_session AS (
        SELECT gamesession_id
        FROM eligible
        GROUP BY gamesession_id
        ORDER BY MAX(TRY_CAST(created_at AS TIMESTAMP)) DESC, gamesession_id DESC
        LIMIT 1
    ),
    selected AS (
        SELECT
            e.*,
            ROW_NUMBER() OVER (
                ORDER BY segment_start, segment_end
            ) AS segment_index
        FROM eligible e
        JOIN latest_session l USING (gamesession_id)
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        lang_detected AS language_detected
    FROM selected
    ORDER BY segment_index
    LIMIT {SEGMENTS_NEEDED}
    """
).df()

def format_timestamp(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

english_segments["segment_timestamp"] = (
    english_segments["segment_start"].map(format_timestamp)
    + " - "
    + english_segments["segment_end"].map(format_timestamp)
)

english_segments["VERDICT"] = ""

english_segments = english_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(english_segments)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268501,https://www.twitch.tv/videos/2854311727,1,00:00:14 - 00:00:15,"Don't be talking, shut my door.",en,
1,141268501,https://www.twitch.tv/videos/2854311727,2,00:02:23 - 00:02:25,We'll start with the basics.,en,
2,141268501,https://www.twitch.tv/videos/2854311727,3,00:02:46 - 00:02:49,actually showed up. You guys owe me ten bucks.,en,
3,141268501,https://www.twitch.tv/videos/2854311727,4,00:02:50 - 00:02:54,"personal. It's just, uh, we're stuck in Groundhog Day out here.",en,
4,141268501,https://www.twitch.tv/videos/2854311727,5,00:02:55 - 00:02:59,"trying to, you know, not go crazy..",en,
5,141268501,https://www.twitch.tv/videos/2854311727,6,00:02:59 - 00:03:06,"Okay, it's simple really, shout next and the guys will let the survivor in.",en,
6,141268501,https://www.twitch.tv/videos/2854311727,7,00:03:20 - 00:03:26,"Next survivor. First, do a simple inspection.",en,
7,141268501,https://www.twitch.tv/videos/2854311727,8,00:03:27 - 00:03:28,Take the flashlight from the table.,en,
8,141268501,https://www.twitch.tv/videos/2854311727,9,00:03:44 - 00:03:47,Take your time. Match what you see against the symptom chart.,en,
9,141268501,https://www.twitch.tv/videos/2854311727,10,00:03:48 - 00:03:55,devil's in the details. Identify the particular ailment and click symptom locked.,en,


In [ ]:
import duckdb
import pandas as pd

LANGUAGE = "en"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

english_segments = con.execute(
    f"""
    WITH latest_session AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected = '{LANGUAGE}'
            AND transcript_segments IS NOT NULL
        ORDER BY
            TRY_CAST(created_at AS TIMESTAMP) DESC,
            gamesession_id DESC
        LIMIT 1
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM latest_session
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            lang_detected AS language_detected
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
            AND len(segment.words) >= {MIN_WORDS}
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        language_detected
    FROM eligible
    WHERE
        segment_start IS NOT NULL
        AND segment_end IS NOT NULL
    ORDER BY segment_index ASC
    LIMIT {SEGMENTS_NEEDED}
    """
).df()


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


english_segments["segment_timestamp"] = (
    english_segments["segment_start"].map(format_timestamp)
    + " - "
    + english_segments["segment_end"].map(format_timestamp)
)

english_segments["VERDICT"] = ""

english_segments = english_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(english_segments)

In [12]:
import duckdb
import pandas as pd

LANGUAGE = "de"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

german_segments = con.execute(
    f"""
    WITH latest_session AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected = '{LANGUAGE}'
            AND transcript_segments IS NOT NULL
        ORDER BY
            TRY_CAST(created_at AS TIMESTAMP) DESC,
            gamesession_id DESC
        LIMIT 1
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM latest_session
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            lang_detected AS language_detected
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
            AND len(segment.words) >= {MIN_WORDS}
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        language_detected
    FROM eligible
    WHERE
        segment_start IS NOT NULL
        AND segment_end IS NOT NULL
    ORDER BY segment_index ASC
    LIMIT {SEGMENTS_NEEDED}
    """
).df()


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


german_segments["segment_timestamp"] = (
    german_segments["segment_start"].map(format_timestamp)
    + " - "
    + german_segments["segment_end"].map(format_timestamp)
)

german_segments["VERDICT"] = ""

german_segments = german_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(german_segments)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141192575,https://www.twitch.tv/videos/2852368077,1,00:00:07 - 00:00:10,"Der kann mich gar nicht erwischt, nie im Leben.",de,
1,141192575,https://www.twitch.tv/videos/2852368077,2,00:00:14 - 00:00:15,ich hab's gerade gesehen.,de,
2,141192575,https://www.twitch.tv/videos/2852368077,3,00:00:25 - 00:00:27,"Eigentlich hat das Gebäude, als wir offiziell in der Zwiebel waren.",de,
3,141192575,https://www.twitch.tv/videos/2852368077,4,00:00:46 - 00:00:53,"Ich seh' halt nicht, keine Ahnung, wo der ist.",de,
4,141192575,https://www.twitch.tv/videos/2852368077,5,00:00:59 - 00:01:04,"Irgendwo Richtung Turm drin, irgendwo da hinten, so ein bisschen weiter links.",de,
5,141192575,https://www.twitch.tv/videos/2852368077,6,00:01:08 - 00:01:16,"einen haben wir am Tachel, einer ist noch unten drin ja ja da snipet er die Lachen",de,
6,141192575,https://www.twitch.tv/videos/2852368077,7,00:01:19 - 00:01:28,"traurige ja gib mir ein Bredi ja ein richtiger Lack jetzt habe ich gepisst das, ich hätte dich geholt",de,
7,141192575,https://www.twitch.tv/videos/2852368077,8,00:01:30 - 00:01:41,"ich weiß nicht, hier haben die die selber gibt's das das war das, was du deinen eigenen Strömen liebst warte mal ich schau mal Das ist jetzt Beste.",de,
8,141192575,https://www.twitch.tv/videos/2852368077,10,00:02:06 - 00:02:13,"Die hat ja noch einer, jetzt können sie Tank und Psych nochmal.",de,
9,141192575,https://www.twitch.tv/videos/2852368077,11,00:02:14 - 00:02:18,"Oh nein, ja er snipet immer noch wo da oben.",de,


In [13]:
import duckdb
import pandas as pd

LANGUAGE = "fr"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

french_segments = con.execute(
    f"""
    WITH latest_session AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected = '{LANGUAGE}'
            AND transcript_segments IS NOT NULL
        ORDER BY
            TRY_CAST(created_at AS TIMESTAMP) DESC,
            gamesession_id DESC
        LIMIT 1
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM latest_session
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            lang_detected AS language_detected
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
            AND len(segment.words) >= {MIN_WORDS}
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        language_detected
    FROM eligible
    WHERE
        segment_start IS NOT NULL
        AND segment_end IS NOT NULL
    ORDER BY segment_index ASC
    LIMIT {SEGMENTS_NEEDED}
    """
).df()


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


french_segments["segment_timestamp"] = (
    french_segments["segment_start"].map(format_timestamp)
    + " - "
    + french_segments["segment_end"].map(format_timestamp)
)

french_segments["VERDICT"] = ""

french_segments = french_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(french_segments)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141267561,https://www.twitch.tv/videos/2854116559,2,00:00:52 - 00:00:55,What you guys doing ?,fr,
1,141267561,https://www.twitch.tv/videos/2854116559,6,00:02:38 - 00:02:41,What you guys doing ?,fr,
2,141267561,https://www.twitch.tv/videos/2854116559,7,00:03:06 - 00:03:14,"Allez mec Attendez Yo Damien, t le premier ?",fr,
3,141267561,https://www.twitch.tv/videos/2854116559,8,00:03:14 - 00:03:24,"Ouais sur tiktak t le premier, même sur Twitch je pense En tout cas le premier message Euh, y'en perd pas de temps Aujourd'hui on essaie d'avancer putain de capa de merde Comme ça c'est fait, il nous manquera quoi ?",fr,
4,141267561,https://www.twitch.tv/videos/2854116559,9,00:03:24 - 00:03:34,"Là, il'on y s'extraire avec l'autre un, ça nous donnera la peluche, et après il manquera juste l'achantique, et problème de l'achantique, c que je sais pas comment je vais le pour la trouver, mais on va tenter, on va tenter.",fr,
5,141267561,https://www.twitch.tv/videos/2854116559,10,00:03:34 - 00:03:38,"Yo Pilouf,'as va ou quoi ? Comment va le Pilouf ?",fr,
6,141267561,https://www.twitch.tv/videos/2854116559,11,00:03:39 - 00:03:44,"Allez, ça part comme ça les gars. Un second. Ça raconte quoi Pilouf ?",fr,
7,141267561,https://www.twitch.tv/videos/2854116559,12,00:03:44 - 00:03:48,"J'espère que tu vas bien Damien. Ah d'ailleurs, je voulais te faire un truc.",fr,
8,141267561,https://www.twitch.tv/videos/2854116559,13,00:04:25 - 00:04:35,"Ah bah tranquille mon Pilouf. J'peux pas t'entendre, j'suis au taf, moi un pouce avec un grand sourire pour me dire bonjour La petit pouce Putain, travailler un dimanche, terrible",fr,
9,141267561,https://www.twitch.tv/videos/2854116559,14,00:04:50 - 00:04:54,"Mais putain, Pilouf alors Pilouf tu reprends lundi, enfin demain les cours",fr,


In [14]:
import duckdb
import pandas as pd

LANGUAGE = "fr"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

french_segments = con.execute(
    f"""
    WITH latest_session AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected = '{LANGUAGE}'
            AND transcript_segments IS NOT NULL
        ORDER BY
            TRY_CAST(created_at AS TIMESTAMP) DESC,
            gamesession_id DESC
        LIMIT 1
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM latest_session
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            lang_detected AS language_detected
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
            AND len(segment.words) >= {MIN_WORDS}
    ),
    deduplicated AS (
        SELECT *
        FROM eligible
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY LOWER(TRIM(segment_text))
            ORDER BY segment_index ASC
        ) = 1
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        language_detected
    FROM deduplicated
    WHERE
        segment_start IS NOT NULL
        AND segment_end IS NOT NULL
    ORDER BY segment_index ASC
    LIMIT {SEGMENTS_NEEDED}
    """
).df()


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


french_segments["segment_timestamp"] = (
    french_segments["segment_start"].map(format_timestamp)
    + " - "
    + french_segments["segment_end"].map(format_timestamp)
)

french_segments["VERDICT"] = ""

french_segments = french_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(french_segments)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141267561,https://www.twitch.tv/videos/2854116559,2,00:00:52 - 00:00:55,What you guys doing ?,fr,
1,141267561,https://www.twitch.tv/videos/2854116559,7,00:03:06 - 00:03:14,"Allez mec Attendez Yo Damien, t le premier ?",fr,
2,141267561,https://www.twitch.tv/videos/2854116559,8,00:03:14 - 00:03:24,"Ouais sur tiktak t le premier, même sur Twitch je pense En tout cas le premier message Euh, y'en perd pas de temps Aujourd'hui on essaie d'avancer putain de capa de merde Comme ça c'est fait, il nous manquera quoi ?",fr,
3,141267561,https://www.twitch.tv/videos/2854116559,9,00:03:24 - 00:03:34,"Là, il'on y s'extraire avec l'autre un, ça nous donnera la peluche, et après il manquera juste l'achantique, et problème de l'achantique, c que je sais pas comment je vais le pour la trouver, mais on va tenter, on va tenter.",fr,
4,141267561,https://www.twitch.tv/videos/2854116559,10,00:03:34 - 00:03:38,"Yo Pilouf,'as va ou quoi ? Comment va le Pilouf ?",fr,
5,141267561,https://www.twitch.tv/videos/2854116559,11,00:03:39 - 00:03:44,"Allez, ça part comme ça les gars. Un second. Ça raconte quoi Pilouf ?",fr,
6,141267561,https://www.twitch.tv/videos/2854116559,12,00:03:44 - 00:03:48,"J'espère que tu vas bien Damien. Ah d'ailleurs, je voulais te faire un truc.",fr,
7,141267561,https://www.twitch.tv/videos/2854116559,13,00:04:25 - 00:04:35,"Ah bah tranquille mon Pilouf. J'peux pas t'entendre, j'suis au taf, moi un pouce avec un grand sourire pour me dire bonjour La petit pouce Putain, travailler un dimanche, terrible",fr,
8,141267561,https://www.twitch.tv/videos/2854116559,14,00:04:50 - 00:04:54,"Mais putain, Pilouf alors Pilouf tu reprends lundi, enfin demain les cours",fr,
9,141267561,https://www.twitch.tv/videos/2854116559,15,00:05:08 - 00:05:15,"Non la semaine pro, putain ok, j'pensais que vous repreniez, mais comment ça se fait La France ils reprennent pas demain ?",fr,


In [15]:
import duckdb
import pandas as pd

LANGUAGE = "pt"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

portuguese_segments = con.execute(
    f"""
    WITH latest_session AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected = '{LANGUAGE}'
            AND transcript_segments IS NOT NULL
        ORDER BY
            TRY_CAST(created_at AS TIMESTAMP) DESC,
            gamesession_id DESC
        LIMIT 1
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM latest_session
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            lang_detected AS language_detected
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
            AND len(segment.words) >= {MIN_WORDS}
    ),
    deduplicated AS (
        SELECT *
        FROM eligible
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY LOWER(TRIM(segment_text))
            ORDER BY segment_index ASC
        ) = 1
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        language_detected
    FROM deduplicated
    WHERE
        segment_start IS NOT NULL
        AND segment_end IS NOT NULL
    ORDER BY segment_index ASC
    LIMIT {SEGMENTS_NEEDED}
    """
).df()


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


portuguese_segments["segment_timestamp"] = (
    portuguese_segments["segment_start"].map(format_timestamp)
    + " - "
    + portuguese_segments["segment_end"].map(format_timestamp)
)

portuguese_segments["VERDICT"] = ""

portuguese_segments = portuguese_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(portuguese_segments)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268410,https://www.twitch.tv/videos/2854267348,1,00:09:52 - 00:09:59,FUERTA DE RAHIM Ta,pt,
1,141268410,https://www.twitch.tv/videos/2854267348,2,00:10:54 - 00:11:10,de pé E Fui FUERTA RAHIM E ASSISTA FUERTA FAZ FUERTA RAHIM FUERTA RAHIM,pt,
2,141268410,https://www.twitch.tv/videos/2854267348,3,00:11:13 - 00:11:19,FUERTA RAHIM porta aberta e isso,pt,
3,141268410,https://www.twitch.tv/videos/2854267348,4,00:11:23 - 00:11:47,ok então estou bebendo um café hoje seria o dia do chill stream mas essa semana eu não joguei a LoL eu queria jogar a LoL eu queria jogar a LoL o chill stream significa que eu decido então se eu quero jogar LoL eu a LoL e vocês pegam no cu aquele eu mais alto talvez,pt,
4,141268410,https://www.twitch.tv/videos/2854267348,5,00:11:54 - 00:11:57,Como você Como você Galaxi?,pt,
5,141268410,https://www.twitch.tv/videos/2854267348,6,00:11:58 - 00:12:02,"Você fazendo tudo, você está certo. Como você Pasta?",pt,
6,141268410,https://www.twitch.tv/videos/2854267348,7,00:12:02 - 00:12:09,"Como você está? Shadow? Tudo bem, garotos? Razy, como você Tchau garotos, tchau a você bem?",pt,
7,141268410,https://www.twitch.tv/videos/2854267348,8,00:12:14 - 00:12:18,"O Urco desde esta manhã, oh Deus, desta manhã...",pt,
8,141268410,https://www.twitch.tv/videos/2854267348,9,00:12:18 - 00:12:21,"Eu não estava. E saiu também no meu schedule, não sei se você viu.",pt,
9,141268410,https://www.twitch.tv/videos/2854267348,10,00:12:22 - 00:12:28,"Que é feito, não mais a Dorari. Mas é a...",pt,


In [16]:
import duckdb
import pandas as pd

LANGUAGE = "es"
SEGMENTS_NEEDED = 25
MIN_WORDS = 4

con = duckdb.connect()

spanish_segments = con.execute(
    f"""
    WITH latest_session AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected = '{LANGUAGE}'
            AND transcript_segments IS NOT NULL
        ORDER BY
            TRY_CAST(created_at AS TIMESTAMP) DESC,
            gamesession_id DESC
        LIMIT 1
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            model_type,
            created_at,
            lang_detected,
            lang_probability,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM latest_session
    ),
    eligible AS (
        SELECT
            gamesession_id,
            url,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            lang_detected AS language_detected
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
            AND len(segment.words) >= {MIN_WORDS}
    ),
    deduplicated AS (
        SELECT *
        FROM eligible
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY LOWER(TRIM(segment_text))
            ORDER BY segment_index ASC
        ) = 1
    )
    SELECT
        gamesession_id,
        url,
        segment_index,
        segment_start,
        segment_end,
        segment_text,
        language_detected
    FROM deduplicated
    WHERE
        segment_start IS NOT NULL
        AND segment_end IS NOT NULL
    ORDER BY segment_index ASC
    LIMIT {SEGMENTS_NEEDED}
    """
).df()


def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


spanish_segments["segment_timestamp"] = (
    spanish_segments["segment_start"].map(format_timestamp)
    + " - "
    + spanish_segments["segment_end"].map(format_timestamp)
)

spanish_segments["VERDICT"] = ""

spanish_segments = spanish_segments[
    [
        "gamesession_id",
        "url",
        "segment_index",
        "segment_timestamp",
        "segment_text",
        "language_detected",
        "VERDICT",
    ]
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(spanish_segments)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,VERDICT
0,141268448,https://www.twitch.tv/videos/2853628924,1,00:03:03 - 00:03:14,hola hola que tal gente silla estamos aquí otra vez un día más vamos al charlando,es,
1,141268448,https://www.twitch.tv/videos/2853628924,2,00:03:17 - 00:03:47,hello como estáis como estáis ahora estoy bastante cansada pero tengo una cosa aquí estamos aquí cumplimos vale pues hoy vamos a jugar a los dinosaurios que me apetece un ratito aunque sea y ya luego vamos al balo en un ratito hoy no me quedaré mucho porque me apetece estar más en la gama todo esto me hecho una siesta cuando he llegado,es,
2,141268448,https://www.twitch.tv/videos/2853628924,3,00:03:47 - 00:03:59,de tres horas y claro una mañana que hay que trabajar entonces prefiero irme tempranito que sabemos como acaba de irse tempranito pero bueno lo cuento igual.,es,
3,141268448,https://www.twitch.tv/videos/2853628924,4,00:04:02 - 00:04:05,eh... Vale. ¿Dónde está? ¿Dónde está?,es,
4,141268448,https://www.twitch.tv/videos/2853628924,6,00:04:11 - 00:04:13,"Let's go. Vamos allá, continúen.",es,
5,141268448,https://www.twitch.tv/videos/2853628924,7,00:04:18 - 00:04:21,"Vale, voy limpiar un momentito rápido el teclado.",es,
6,141268448,https://www.twitch.tv/videos/2853628924,9,00:04:38 - 00:04:49,"Pero no lo que tenía que hacer, tipo, las dos canastas y los ladrillos de piedra vale,",es,
7,141268448,https://www.twitch.tv/videos/2853628924,10,00:04:52 - 00:05:03,"ehm, los de piedra y las dos canastas, tengo que hacer nooo, una planta por la puta,",es,
8,141268448,https://www.twitch.tv/videos/2853628924,11,00:05:08 - 00:05:17,"falta una planta ah ya tengo las dos vale, ok, eh vamos a reunirnos con Bryce",es,
9,141268448,https://www.twitch.tv/videos/2853628924,12,00:05:35 - 00:05:44,"¿Por qué hay emisiones? ¿Qué más tenían? Aumentar la población, pero eso lo vas diciendo poco a",es,
